<a href="https://colab.research.google.com/github/kamimartinez/VMS-prediction/blob/main/DataUnderstanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Conectar con Google Drive e Importar Librerías
Ejecuta la siguiente celda para dar acceso a Colab a tus archivos de Drive e importar `pandas` y `numpy`.

In [45]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Configurar la Ruta y Listar Archivos
Por defecto, al montar Google Drive, tu unidad principal se encuentra en `/content/drive/My Drive/`.

IMPORTANTE => Hacer un acceso directo de la carpeta Datos a tu unidad para que el programa pueda leer los archivos.


In [46]:
import os
import pandas as pd

def buscar_ruta_carpeta(nombre_carpeta_buscar, ruta_principal = '/content/drive/My Drive/Datos'):
    # Esta función busca de manera insensible a mayúsculas/minúsculas
    nombre_buscar_lower = nombre_carpeta_buscar.lower()

    for raiz, directorios, archivos in os.walk(ruta_principal):
        for directorio in directorios:
            if directorio.lower() == nombre_buscar_lower:
                return os.path.join(raiz, directorio)
    return None

nombre_carpeta = 'datos vacas en establo marzo_junio_2025'
ruta_final = buscar_ruta_carpeta(nombre_carpeta)

if ruta_final:
    print(f"¡Éxito! Carpeta detectada en: {ruta_final}\n")
    archivos = [f for f in os.listdir(ruta_final) if f.endswith('.csv')]
    print(f"Se encontraron {len(archivos)} archivos de datos relevantes:")
    for i, archivo in enumerate(archivos[:10], 1):
        print(f"{i}. {archivo}")
    if len(archivos) > 10:
        print("... y más.")
else:
    print("Aún no se detectó el acceso directo. Por favor verifica que esté en 'Mi Unidad/Datos/datos vacas en establo marzo_junio_2025' y vuelve a ejecutar esta celda.")

¡Éxito! Carpeta detectada en: /content/drive/My Drive/Datos/DATOS vacas en establo marzo_junio_2025

Se encontraron 37 archivos de datos relevantes:
1. 1613.csv
2. 1233.csv
3. 1624.csv
4. 6238.csv
5. 2076.csv
6. 6191.csv
7. 1221.csv
8. 2078.csv
9. 2082.csv
10. 1554.csv
... y más.


### 4. Lectura, Consolidación y Ordenamiento de los Datos
El siguiente código leerá todos los archivos de datos encontrados en la carpeta, omitirá la primera fila de datos de cada uno (manteniendo la estructura de cabeceras), combinará toda la información en un solo DataFrame de Pandas y finalmente ordenará los registros por la columna **'Hora de inicio'**.

In [47]:
import os
import pandas as pd

lista_df = []

print("Iniciando la lectura de archivos...")
for archivo in archivos:
    # Omitir archivos que no empiecen con un número
    if not archivo[0].isdigit():
        continue

    ruta_archivo = os.path.join(ruta_final, archivo)

    try:
        # Leer según el tipo de extensión del archivo
        # skiprows=1 se usa para saltarse la primera línea de datos físicos del archivo.
        df_temp = pd.read_csv(ruta_archivo, skiprows=1)

        # Añadir una columna para saber de qué archivo proviene cada registro
        df_temp['Origen_Archivo'] = archivo
        lista_df.append(df_temp)

    except Exception as e:
        print(f"No se pudo procesar el archivo {archivo}. Error: {e}")

print(f"\nArchivos procesados correctamente: {len(lista_df)}")
if lista_df:
    print(f"Columnas que deberían de tener: {len(lista_df[0].columns)}")
    acum = {'sum': 0, 'max': 0, 'min': 9999}
    for df in (lista_df):
      acum["sum"] += len(df.columns)
      acum["max"] = max(acum["max"], len(df.columns))
      acum["min"] = min(acum["min"], len(df.columns))

    print(f"Promedio de categorías: {acum['sum'] / len(lista_df)}")
    print(f"Máximo de categorías:  {acum['max']}")
    print(f"Mínimo de categorías: {acum['min']}")

Iniciando la lectura de archivos...

Archivos procesados correctamente: 34
Columnas que deberían de tener: 36
Promedio de categorías: 36.0
Máximo de categorías:  36
Mínimo de categorías: 36


In [48]:
# Concatenar todos los dataframes en uno solo
if lista_df:
    df_consolidado = pd.concat(lista_df, ignore_index=True)
    print(f"\n¡Unión completada! Total de registros cargados: {len(df_consolidado)}")

    # Intentar ordenar por 'Hora de inicio' si la columna existe
    columna_orden = 'Hora de inicio'
    if columna_orden in df_consolidado.columns:
        # Convertimos a tipo datetime para ordenar cronológicamente de forma correcta
        df_consolidado[columna_orden] = pd.to_datetime(df_consolidado[columna_orden], errors='coerce')
        df_consolidado = df_consolidado.sort_values(by=columna_orden).reset_index(drop=True)
        print(f"Datos ordenados correctamente por '{columna_orden}'.")
    else:
        print(f"\nAdvertencia: No se encontró la columna '{columna_orden}' para ordenar. Las columnas disponibles son:")
        print(list(df_consolidado.columns))

    # Mostrar una vista previa del DataFrame final
    display(df_consolidado.head())
else:
    print("No se cargó ningún dato. Verifica que los archivos no estén vacíos o corruptos.")


¡Unión completada! Total de registros cargados: 7249


/tmp/ipykernel_3066/3493702968.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_consolidado[columna_orden] = pd.to_datetime(df_consolidado[columna_orden], errors='coerce')


Datos ordenados correctamente por 'Hora de inicio'.


,Hora de inicio,Acción,Duración (mm:ss),Producción (kg),Número de ordeño,RCS (* 1000 células / ml),Patada,Incompleto,Pezones no encontrados,Ubre,...,Razón de la desviación,DI.3,DD.3,TI.3,TD.3,DI.4,DD.4,TI.4,TD.4,Origen_Archivo
0,2025-01-03 00:06:00,Ordeño,06:54,16.30,1.0,NaN,NaN,NaN,NaN,0.0,...,NaN,1.50,1.62,1.68,1.74,3.77,3.25,5.03,4.25,1624.csv
1,2025-01-03 00:16:00,Ordeño,08:58,14.48,1.0,NaN,TI,NaN,NaN,0.0,...,NaN,1.08,1.38,1.20,1.26,3.07,2.75,4.31,4.35,1613.csv
2,2025-01-03 00:49:00,Ordeño,10:28,7.28,1.0,NaN,DI,"TI,TD",TD,1.0,...,NaN,1.92,2.40,1.98,0.00,2.34,4.23,0.71,0.00,6178.csv
3,2025-01-03 01:06:00,Ordeño,08:29,20.64,1.0,NaN,NaN,NaN,NaN,0.0,...,NaN,1.68,1.62,1.50,1.86,4.62,3.88,6.17,5.97,6194.csv
4,2025-01-03 01:37:00,Ordeño,07:55,23.08,1.0,NaN,NaN,NaN,NaN,0.0,...,NaN,1.80,2.22,2.82,2.10,2.58,3.29,8.63,8.58,6238.csv


### 5. Procesamiento de Nuevos Datos Históricos (Clasificación por tipo de archivo)
En esta sección clasificamos los archivos `.xlsx` de la carpeta `Datos vacas historico hasta enero 2026/TEC-UAQ-UNAM` en diferentes DataFrames según su prefijo.

In [49]:
import os
import pandas as pd

# 1. Definir y buscar la nueva carpeta de datos históricos
nombre_carpeta = 'TEC-UAQ-UNAM'
ruta_historico = buscar_ruta_carpeta(nombre_carpeta, ruta_principal='/content/drive/My Drive/Datos/Datos vacas historico hasta enero 2026')

def read_detailed_data():
  if ruta_historico:
      print(f"¡Éxito! Carpeta detectada en: {ruta_historico}\n")

      # Listar todos los archivos Excel (.xlsx)
      todos_archivos = [f for f in os.listdir(ruta_historico) if f.endswith('.xlsx') or f.endswith('.xls')]
      print(f"Se encontraron {len(todos_archivos)} archivos Excel en total.")

      # Inicializar listas para clasificar los DataFrames
      lista_concentrado = []
      lista_produccion_leche = []

      # 2. Iterar y clasificar los archivos
      for archivo in todos_archivos:
          ruta_archivo = os.path.join(ruta_historico, archivo)

          # Extraer el ID o número del animal del nombre del archivo como referencia
          # Ej: 'Concentrado consumido 216.xlsx' -> extrae '216'
          id_vaca = "".join(filter(str.isdigit, archivo))

          try:
              if archivo.lower().startswith('concentrado consumido'):
                  df_temp = pd.read_excel(ruta_archivo)
                  df_temp['Origen_Archivo'] = archivo
                  df_temp['ID_Vaca'] = id_vaca if id_vaca else 'Desconocido'
                  lista_concentrado.append(df_temp)

              else:
                  df_temp = pd.read_excel(ruta_archivo, skiprows=1)
                  df_temp['Origen_Archivo'] = archivo
                  df_temp['ID_Vaca'] = id_vaca if id_vaca else 'Desconocido'
                  lista_produccion_leche.append(df_temp)

          except Exception as e:
              print(f"Error al procesar el archivo {archivo}: {e}")

      print(f"\nClasificación completada:")
      print(f"- Archivos de 'Concentrado consumido' procesados: {len(lista_concentrado)}")
      print(f"- Archivos de 'Producciones de leche por sesión' procesados: {len(lista_produccion_leche)}")

      return lista_concentrado, lista_produccion_leche

  else:
      print(f"No se encontró la carpeta '{nombre_carpeta}'. Asegúrate de que esté dentro de 'Mi Unidad/Datos/'.")


### 6. Consolidación de los nuevos DataFrames
Creamos los DataFrames integrados para cada tipo de dato y visualizamos su estructura.

In [50]:
def show_details(lista_concentrado, lista_produccion_leche):
  # Consolidar Concentrado Consumido
  if lista_concentrado:
      concentradoConsumido = pd.concat(lista_concentrado, ignore_index=True)
      print(f"¡DataFrame 'concentradoConsumido' creado exitosamente!")
      print(f"Total de filas: {len(concentradoConsumido)} | Columnas: {list(concentradoConsumido.columns)}\n")
      display(concentradoConsumido.head())
  else:
      print("No se encontraron archivos de Concentrado Consumido.")

  print("-" * 80)

  # Consolidar Producción de Leche por Sesión
  if lista_produccion_leche:
      produccionesLecheSesion = pd.concat(lista_produccion_leche, ignore_index=True)
      print(f"¡DataFrame 'produccionesLecheSesion' creado exitosamente!")
      print(f"Total de filas: {len(produccionesLecheSesion)} | Columnas: {list(produccionesLecheSesion.columns)}\n")
      display(produccionesLecheSesion.head())
  else:
      print("No se encontraron archivos de Producción de Leche.")


In [51]:
lista_concentrado, lista_produccion_leche = read_detailed_data()

show_details(lista_concentrado, lista_produccion_leche)

¡Éxito! Carpeta detectada en: /content/drive/My Drive/Datos/Datos vacas historico hasta enero 2026/TEC-UAQ-UNAM

Se encontraron 404 archivos Excel en total.

Clasificación completada:
- Archivos de 'Concentrado consumido' procesados: 203
- Archivos de 'Producciones de leche por sesión' procesados: 201
¡DataFrame 'concentradoConsumido' creado exitosamente!
Total de filas: 130315 | Columnas: ['Alimento', 'Hora evento', 'Consumido', 'Consumido Mat. Seca', 'Origen_Archivo', 'ID_Vaca']



,Alimento,Hora evento,Consumido,Consumido Mat. Seca,Origen_Archivo,ID_Vaca
0,Feed1,2026-01-28 10:00:00,1.164,1.164,Concentrado consumido 1231.xlsx,1231
1,Feed1,2026-01-27 23:00:00,0.837,0.837,Concentrado consumido 1231.xlsx,1231
2,Feed1,2026-01-27 12:00:00,2.099,2.099,Concentrado consumido 1231.xlsx,1231
3,Feed1,2026-01-26 10:00:00,1.625,1.625,Concentrado consumido 1231.xlsx,1231
4,Feed1,2026-01-25 21:00:00,1.532,1.532,Concentrado consumido 1231.xlsx,1231


--------------------------------------------------------------------------------
¡DataFrame 'produccionesLecheSesion' creado exitosamente!
Total de filas: 278968 | Columnas: ['Hora de inicio', 'Número de ordeño', 'Duración (mm:ss)', 'Producción (kg)', 'RCS (* 1000 células / ml)', 'Intervalo de ordeño (hh:mm)', 'DI', 'DD', 'TI', 'TD', 'Ubre', 'Pezón', 'DI.1', 'DD.1', 'TI.1', 'TD.1', 'DI.2', 'DD.2', 'TI.2', 'TD.2', 'DI.3', 'DD.3', 'TI.3', 'TD.3', 'DI.4', 'DD.4', 'TI.4', 'TD.4', 'Ubre.1', 'DI.5', 'DD.5', 'TI.5', 'TD.5', 'Destino Leche', 'Razón de la desviación', 'Programa de lavado', 'MS', 'Usuario', 'Patada', 'Incompleto', 'Pezones no encontrados', 'MDI', 'Origen_Archivo', 'ID_Vaca']



,Hora de inicio,Número de ordeño,Duración (mm:ss),Producción (kg),RCS (* 1000 células / ml),Intervalo de ordeño (hh:mm),DI,DD,TI,TD,...,Razón de la desviación,Programa de lavado,MS,Usuario,Patada,Incompleto,Pezones no encontrados,MDI,Origen_Archivo,ID_Vaca
0,2026-01-28 06:38:06,1,00:04:23,7.66,NaN,07:19:29,0.47,1.69,2.72,2.78,...,NaN,NaN,VMS 3,NaN,NaN,NaN,NaN,1.241349,Producciones de leche por sesión 1216.xlsx,1216
1,2026-01-27 23:14:31,3,00:04:06,8.62,NaN,08:38:42,0.55,1.85,3.05,3.17,...,NaN,NaN,VMS 3,NaN,NaN,NaN,NaN,1.141168,Producciones de leche por sesión 1216.xlsx,1216
2,2026-01-27 14:31:06,2,00:04:43,12.18,NaN,11:40:13,1.08,2.62,4.14,4.34,...,NaN,NaN,VMS 3,NaN,NaN,NaN,NaN,1.127622,Producciones de leche por sesión 1216.xlsx,1216
3,2026-01-27 02:45:59,1,00:04:54,10.62,NaN,12:32:29,0.55,2.40,3.78,3.89,...,NaN,NaN,VMS 3,NaN,NaN,NaN,NaN,1.145002,Producciones de leche por sesión 1216.xlsx,1216
4,2026-01-26 14:08:35,2,00:04:55,11.70,NaN,11:33:52,0.70,2.62,4.10,4.28,...,NaN,NaN,VMS 3,NaN,NaN,NaN,NaN,1.150731,Producciones de leche por sesión 1216.xlsx,1216
